In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import date

In [0]:
# scopes = dbutils.secrets.listScopes()
# for scope in scopes:
#     print(f"Scope name: {scope.name}")

In [0]:

SCOPE   = "pnp-secert-scope"   
client_id     = dbutils.secrets.get(SCOPE, "pnp-client-id")
client_secret = dbutils.secrets.get(SCOPE, "pnp-client-secret")
tenant_id     = dbutils.secrets.get(SCOPE, "pnp-tenant-id")


spark.conf.set("fs.azure.account.auth.type.pnpbyedge.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.pnpbyedge.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.pnpbyedge.dfs.core.windows.net",client_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.pnpbyedge.dfs.core.windows.net", client_secret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.pnpbyedge.dfs.core.windows.net", f"https://login.windows.net/{tenant_id}/oauth2/token")

In [0]:


today = date.today().strftime('%Y-%m-%d')
path = f'abfss://brozen@pnpbyedge.dfs.core.windows.net/final/{today}/*.csv'

# Read all files in today's folder
df = spark.read\
    .format('csv')\
    .option('header', True)\
    .option('inferSchema', True)\
    .load(path)

In [0]:
df = df.select(
col('retailer_name'),
col('sku'),
col('upc'),
col('product_id'),
col('product_name'),
col('unit_price').alias('price'), 
col('brand_id'),
col('manufacturer_id'), 
col('vertical').alias('main_category'),
col('category_l1'), 
col('category_l2'), 
col('category_l3'),
col('currency'), 
col('pack_size'), 
col('product_url'),
col('country'), 
col('is_promo'), 
col('promo_price'),
col("activity_log"))

In [0]:
df = df.filter(
    (upper(col('country').isin(['IE','GB','UK'])))\
    & (col('sku').isNotNull())\
    & (col("product_name").isNotNull())\
    & (col("price").isNotNull())\
    & (col("currency").isNotNull())\
    & (col('pack_size').isNotNull())\
    
    )

In [0]:
df = df.withColumn('date', date_add(current_date(),0))

In [0]:
df = df.withColumn('price',
regexp_replace(
    regexp_replace(
        trim(col("price").cast('string')),
    '£',''),
'GBP',"").cast(DoubleType())              
  )

In [0]:
def clean_col(df, col_name):
    return df.withColumn(col_name, 
                              regexp_replace(
                                  upper(trim(col(col_name)).cast('string')), 
                                 
                          '-','')
                          )
    
df = clean_col(df, "sku")
df = clean_col(df, "product_id")
df = clean_col(df, "brand_id")
df = clean_col(df, "manufacturer_id")

# Verify
df.select('sku', 'upc', 'product_id', 
          'brand_id', 'manufacturer_id')

In [0]:
df = df.withColumn('category_series',
              concat_ws(">>",
                        col("main_category"),
                        col("category_l1"),
                        col("category_l2"),
                        col("category_l3")
                        )

)

In [0]:
df = df.withColumn('is_promo',
            when(lower(trim(col('is_promo').cast("string"))).isin(['yes','true','1','y','t']), True)\
            .when(lower(trim(col('is_promo').cast("string"))).isin(['no','false','0','n','f']), False)\
            .when(col("is_promo").isNull(), False)\
            .when(col('promo_price') >= col('price'), None)\
            .when(col('is_promo') == False, None)
            .otherwise(col("is_promo")).cast("boolean")
            )
              

In [0]:
df = df.withColumn("currency", 
                         when(lower(trim(col("currency").cast('string'))).isin(['gbp', '£']), '£')\
                        .when(lower(trim(col("currency").cast('string'))).isin(['eur', '€']), '€')\
                        .when(lower(trim(col("country").cast('string'))).isin(['gb', 'uk']), '£')\
                        .when(lower(trim(col("country").cast('string'))).isin(['ie', 'fr']), '€')\
                        .otherwise(col("currency")
                         )
            
            )

In [0]:
df = df.withColumn('price',
            when(col("price")<= 0.20, None)\
            .otherwise(col("price"))
             )


In [0]:

today = date.today().strftime('%Y-%m-%d')

df.write.format('parquet').mode('overwrite')\
    .option('path', f'abfss://silver@pnpbyedge.dfs.core.windows.net/{today}/')\
    .save()